In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [ ]:
# select device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
# architecture
class EmbedWipeout(nn.Module):
    def __init__(self, h=28, w=28, n=28*28, l=10):
        super().__init__()
        self.embed = nn.Sequential(
            nn.Flatten(),
            nn.Linear(h*w, n),
        )
        self.norm = nn.LayerNorm(n)
        self.wipeout = nn.Linear(n, l, bias=False)

    def forward(self, x):
        hidden = self.norm(self.embed(x))
        logits = self.wipeout(hidden)
        if self.training:
            return logits
        else:
            return F.softmax(logits, dim=1)

In [ ]:
# create classifier
model = EmbedWipeout(28,28,3,4).to(device)

In [ ]:
# get datasets
train_dataset = datasets.MNIST(root='data', train=True, download=True, transform=transforms.ToTensor())
test_dataset = datasets.MNIST(root='data', train=False, download=True, transform=transforms.ToTensor())

In [ ]:
# select a subset of classes
classes_to_keep = {0, 6, 8, 9}
class_to_id = {class_: i for i, class_ in enumerate(classes_to_keep)}
print(class_to_id)
id_to_class = {i: class_ for class_, i in class_to_id.items()}
print(id_to_class)

In [ ]:
class RemappedSubset(Dataset):
    def __init__(self, original_dataset, indices, class_to_new):
        self.original = original_dataset
        self.indices = indices
        self.class_to_new = class_to_new

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        x, y = self.original[real_idx]
        new_y = self.class_to_new[int(y)]
        return x, new_y

In [ ]:
# create subset
train_indices = [i for i, (_, label) in enumerate(train_dataset) if label in classes_to_keep] # indices for train subset
test_indices = [i for i, (_, label) in enumerate(test_dataset) if label in classes_to_keep] # indices for test subset
train_subset = RemappedSubset(train_dataset, train_indices, class_to_id)
test_subset = RemappedSubset(test_dataset, test_indices, class_to_id)

In [ ]:
# test subset
labels = torch.tensor([label for _, label in train_subset])
print(torch.unique(labels))
labels = torch.tensor([label for _, label in test_subset])
print(torch.unique(labels))

In [ ]:
# create train and test dataloader
batch_size = 512
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)

In [ ]:
# Define loss funcion
criterion = nn.CrossEntropyLoss()

In [ ]:
# Define optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# scheduler
step_size = 16
gamma = 0.7
scheduler = StepLR(optimizer, step_size=step_size, gamma=gamma)

In [ ]:
# training
history_loss = []
history_acc = []
history_test_acc = []
epoch = 0

In [ ]:
def train(num_epochs):
    global epoch
    for _ in range(num_epochs):
        # change model in training mood
        model.train()

        # to record loss and accuracy
        batch_loss = []
        total_train = 0
        correct_train = 0

        for batch, (x_train, y_train) in enumerate(train_loader):

            # send data to device
            input = x_train.to(device)

            # reset parameters gradient to zero
            optimizer.zero_grad()

            # forward pass to the model
            output = model(input)

            # categorization
            expected_output = y_train.to(device)

            # cross entropy loss
            loss = criterion(output, expected_output)

            # find gradients
            loss.backward()
            # update parameters using gradients
            optimizer.step()

            # recording loss
            batch_loss.append(loss.item())

            # recording accuracy
            total_train += output.shape[0]
            correct_train += torch.argmax(output,dim=1).to('cpu').eq(y_train).sum().item()

        epoch_loss = np.average(batch_loss)
        epoch_acc = (100.0 * correct_train) / total_train

        history_loss.append(epoch_loss)
        history_acc.append(epoch_acc)

        total_test = 0
        correct_test = 0

        model.eval()

        for batch, (x_test, y_test) in enumerate(test_loader):

            # send data to device
            input = x_test.to(device)

            # forward pass to the model
            with torch.no_grad():
                output = model(input)

            # recording accuracy
            total_test += output.shape[0]
            correct_test += torch.argmax(output,dim=1).to('cpu').eq(y_test).sum().item()

        test_acc = (100.0 * correct_test) / total_test

        history_test_acc.append(test_acc)

        print(f'Epoch: {epoch} Loss: {epoch_loss:.6f} Accuracy: {epoch_acc:.4f} Test accuracy: {test_acc:.4f} Learning Rate: {scheduler.get_last_lr()[0]:.7f}')
        scheduler.step()
        epoch += 1

In [ ]:
# training
print(f"number of trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")
num_epochs = 70
train(num_epochs)

In [ ]:
def plot_history():
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))  # 1 row, 2 columns
    # Loss curve
    axes[0].plot(history_loss, 'r', linewidth=3.0)
    axes[0].set_title('Loss Curve', fontsize=16)
    axes[0].set_xlabel('Epochs', fontsize=14)
    axes[0].set_ylabel('Loss', fontsize=14)
    axes[0].legend(['Training Loss'], fontsize=12)
    # Accuracy curves
    axes[1].plot(history_acc, 'r', linewidth=3.0)
    axes[1].plot(history_test_acc, 'b', linewidth=3.0)
    axes[1].set_title('Accuracy Curves', fontsize=16)
    axes[1].set_xlabel('Epochs', fontsize=14)
    axes[1].set_ylabel('Accuracy', fontsize=14)
    axes[1].legend(['Training Accuracy', 'Validation Accuracy'], fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_history()

In [ ]:
model.eval()

In [ ]:
# Save model
def save(model_name):
    torch.save(model.state_dict(), model_name) # weights only
    print('Saved trained model at %s ' % model_name)

In [ ]:
save('mnist_classifier_embed-wipeout.pth')

In [ ]:
# download the model
from google.colab import files
files.download('mnist_classifier_embed-wipeout.pth')

In [ ]:
# instead of training, we can download and load the result
#!wget http://www.agentspace.org/download/mnist_classifier_embed-wipeout.pth
#model = EmbedWipeout(28,28,3,4).to(device)
#model.load_state_dict(torch.load('mnist_classifier_embed-wipeout.pth', weights_only=True))

In [ ]:
# test
sample = train_dataset[train_indices[0]]
print(sample[0].shape,sample[1])

In [ ]:
probabilities = model(sample[0].unsqueeze(0).to(device))
print(probabilities.argmax().item())

In [ ]:
sample_ = sample[0].squeeze(0).detach().numpy()
plt.imshow(sample_, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
# representatives
vecs = model.wipeout.weight.detach().cpu().numpy()
print(vecs.shape)
print(vecs)

In [ ]:
# visualize representatives
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# colormap with one distinct color per vector
colors = plt.cm.tab10(range(len(vecs)))

for i, v in enumerate(vecs):
    x, y, z = v.tolist()
    ax.quiver(0, 0, 0, x, y, z, color=colors[i], label=f"{id_to_class[i]}")
    ax.scatter([x], [y], [z], color=colors[i])

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")

# place legend outside plot for readability
ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1))

plt.tight_layout()
plt.show()


In [ ]:
sample = train_dataset[train_indices[1]]

In [ ]:
embedding = model.norm(model.embed(sample[0].to(device)))[0].detach().cpu().numpy()
print(embedding.shape)
print(embedding)

In [ ]:
embedding @ vecs.T

In [ ]:
id_to_class[np.argmax(embedding @ vecs.T)]

In [ ]:
sample_ = sample[0].squeeze(0).detach().numpy()
plt.imshow(sample_, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
# norm parameters
normw = model.norm.weight.detach().cpu().numpy()
normb = model.norm.bias.detach().cpu().numpy()
# out = normw * (inp-inp.mean()) / torch.sqrt(((inp-inp.mean())**2).mean() + 1e-05) + normb
print(normw)
print(normb)

In [ ]:
# embedding parameters
projection = model.embed[1].weight.detach().cpu().numpy()
bias = model.embed[1].bias.detach().cpu().numpy()
print(projection.shape)
print(bias.shape)

In [ ]:
# back projection
def back(embedding):
    image = ((embedding-normb) / normw - bias) @ np.linalg.pinv(projection.T)
    image = (image-image.min()) / (image.max()-image.min())
    return image.reshape(28,28)

In [ ]:
image = back(vecs[0])

In [ ]:
plt.imshow(image, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
image = back(vecs[1])
plt.imshow(image, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
image = back(vecs[2])
plt.imshow(image, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
image = back(vecs[3])
plt.imshow(image, cmap='gray')
plt.axis('off')
plt.show()